Emmanuel John Barretto

In [ ]:
pip install pandas


In [ ]:
import pandas as pd

In [ ]:
df1 = pd.read_csv("tweets.csv")
df2 = pd.read_csv("tweets2.csv")
df3 = pd.read_csv("tweets3.csv")

In [ ]:
df = pd.concat([df1, df2, df3])

In [ ]:
df.head

In [ ]:
df.to_csv('allTweets.csv', index=False)

In [ ]:
pip install nltk

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import numpy as np

nltk.download('stopwords')
nltk.download('punkt')


In [ ]:
dfTest = pd.read_csv("morleeTweets.csv")

Cleansing of tweets of URLs, #, @ ect, Tokenization and stopwords

In [ ]:
def clean_tweet(tweet):
    cleaned_tweet = re.sub(r'(@[A-Za-z0-9_]+)|[^\w\s]|#|http\S+|RT', '', tweet)
    words = word_tokenize(cleaned_tweet)

    sw = set(stopwords.words('english'))  # Set for faster lookup
    sw.remove('not')  # Exclude 'not' from stopwords
    filtered_words = [word for word in words if word.lower() not in sw]

    return ' '.join(filtered_words)

In [ ]:
dfTest['full_text'] = dfTest['full_text'].apply(clean_tweet)

In [ ]:
dfTest.to_csv('MorleeCleanedTweets.csv', index=False)

Remove Numeric value and duplicates

In [ ]:
def remove_duplicates_and_numbers(text):
    # Remove duplicate words
    words = text.split()
    unique_words = sorted(set(words), key=words.index)
    text = ' '.join(unique_words)
    # Remove numbers
    text = re.sub(r'\d+', '', text)
    return text.strip()

In [ ]:
dfTest['full_text'] = dfTest['full_text'].apply(remove_duplicates_and_numbers)


In [ ]:
dfTest.to_csv('tweetsCleaned1.csv', index=False)

Punctuation Removal

In [ ]:
def remove_punctuation(text):
    return re.sub(r'[^\w\s]', '', text)

In [ ]:
dfTest['full_text'] = dfTest['full_text'].apply(remove_punctuation)

In [ ]:
dfTest.to_csv('tweetsCleaned1.csv', index=False)

Lemmatizer

In [ ]:
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')


In [ ]:
lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(word):
    """Map POS tag to first character lemmatize() accepts"""
    tag = nltk.pos_tag([word])[0][1][0].upper()
    tag_dict = {"J": wordnet.ADJ,
                "N": wordnet.NOUN,
                "V": wordnet.VERB,
                "R": wordnet.ADV}

    return tag_dict.get(tag, wordnet.NOUN)

def lemmatize_text(text):
    words = nltk.word_tokenize(text)
    lemmatized_words = [lemmatizer.lemmatize(word, get_wordnet_pos(word)) for word in words]
    return ' '.join(lemmatized_words)

In [ ]:
dfTest['lemmatized_text'] = dfTest['full_text'].apply(lemmatize_text)

In [ ]:
dfTest.to_csv('MorleeCleanedTweets.csv', index=False)

Starting of the sentiment Vader analysis

In [ ]:
nltk.download('vader_lexicon')
from nltk.sentiment.vader import SentimentIntensityAnalyzer as SIA

In [ ]:
df = pd.read_csv('MorleeCleanedTweets.csv')

Calculate Sentiment

In [ ]:
analyzer = SIA()
df['compound'] = [analyzer.polarity_scores(x)['compound'] for x in df['lemmatized_text']]
df['negative'] = [analyzer.polarity_scores(x)['neg'] for x in df['lemmatized_text']]
df['neutral'] = [analyzer.polarity_scores(x)['neu'] for x in df['lemmatized_text']]
df['positive'] = [analyzer.polarity_scores(x)['pos'] for x in df['lemmatized_text']]

In [ ]:
df['sentiment']='neutral'
df.loc[df.compound>0.05,'sentiment']='positive'
df.loc[df.compound<-0.05,'sentiment']='negative'

In [ ]:
df.to_csv('TweetsSentimentsMorlee.csv', index=False)